In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# B: adopted All-Bit continuation, 0.10 × 2

**Run all** executes the fixed new experiment or resumes its own unfinished units.
Input BASE is the latest resumable-task-v1/run/inputs/terminal_latent.pt.
START is the actual `terminal` field from that task's step_2_unit.pt (ALLBIT_TWO).
The old HALF input, old gradient and planned delta are not new experiment inputs.
The original 0.5 × 2 updates are not repeated. Input paths and SOURCE_REF are editable
locators, without source/version/Git identity gates.

The new independent Drive folder is:
MyDrive/Video-WM/Local-Joint-State-Payload-V1-All-Bit-Continue/resumable-task-v1-2x010/.
The old completed task stays a saved input. Reopening this notebook and Run all
returns to the new task; each attempt's logs and physical calls remain available.
Completed tasks and method-defined zero-gradient/active-tie terminal outcomes only report.

Fixed views: BASE / START_SAVED / CONT_MID / CONT_FINAL.
Primary comparison: CONT_FINAL against START_SAVED, with MID and original BASE retained.
Complete uninterrupted budget: **1 VAE load, 6 decode, 2 decoder VJP**;
184 ordinary chunks, 92 gradient-forward chunks, 92 nominal replay chunks;
**352 windows, 5632 chips, 220 original metrics**, retaining all missing/failed slots.
No encode, DiT, native writer, scheduler, codec, new source or M1/M2.

L = 0.8 mean_32 H_0.02(margin) + 0.2 H_0.01(G).
H_a(x)=0 above a, (a-x)^2/(2a) for 0<x<a, and a/2-x at x<=0.
All 32 terms remain; weak positives and state effort reactivate below their targets.
These targets are optimization preferences, not detection/quality PASS thresholds.
Each new scalar VJP uses its actual START or MID terminal and the existing frozen
FP32 Wan decoder/clamp/readout gradient path. Mask once, normalize in FP64, step 0.10.
Recompute displacement from actual current terminal minus original BASE in FP64,
project into radius <=1, and save the FP32 terminal. Only if FP32 rounding escapes
the ball, round overshooting coordinates one FP32 neighbor toward BASE. Record
projection, rounding, actual radius and scalar predictions. Never refill the radius.
No old gradient reuse, scan, rollback selection, third step, cap expansion or retry to success.

Track all 32 margins, the six original negative bits, the original 26 positive bits,
f3b5 and each view's current weakest positive, G, losses, gained/lost bits and deficits.
Lower loss does not guarantee new positive bits, preserve G or improve quality.
Full 181-frame/180-pair quality, ROI inside/outside residuals, actual peak neighborhoods
and crops are recomputed for these views. Images use RGB[0,1] and residual[-.25,.25];
numeric quality remains unclipped. A BASE reader failure does not remove usable BASE RGB.

Reuse complete input/gradient pairs, delta/actual terminal units, saved RGB, raw,
score and quality. Units commit immediately; every 60 seconds the last committed
recoverable state and progress are saved, retaining the previous readable snapshot.
In-flight decode/VJP cannot resume inside autograd and may need a whole-unit recompute;
every physical attempt, return and failure remains counted beyond the ordinary budget.
Saved MID RGB is scored on CPU before remaining g2/final work. Saved FINAL RGB needs
only CPU postprocessing. Completed raw/score is retained if its RGB is damaged;
missing quality is reported without silently decoding a new replacement observation.
CPU-only tails do not require CUDA or load a VAE. GPU is needed for unfinished model work.

The preceding full source task took 17.05 minutes, peak allocated19.01GiB/reserved21.72GiB;
this is a historical resource observation, not a runtime or memory guarantee for this task.
Current delivery validation is CPU/fake/static. Real Colab/GPU execution is user-run.
One known-message saved-terminal diagnostic establishes no blind decoding, FPR,
codec survival, native/CFG causality or scientific PASS. Return the entire task folder.

In [ ]:
from pathlib import Path
import datetime, json, os, signal, subprocess, sys, time, traceback
SOURCE_REF = 'dev/local-joint-state-payload-v1'  # Editable GitHub branch/tag/ref.
FIXED_CONFIG = {'schema': 'local-joint-all-bit-continue-v1', 'source_run': 'resumable-task-v1-2x010', 'source_arm': 'ALLBIT_TWO', 'model': {'id': 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers', 'revision': '0fad780a534b6463e45facd96134c9f345acfa5b'}, 'carrier': {'key': 'local-joint-state-payload-v1-first-mechanism', 'message_hex': '8001a55a', 'rho': 0.5, 'cap': 1.0}, 'device': 'cuda', 'inputs': {'terminal_latent': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-All-Bit/resumable-task-v1/run/inputs/terminal_latent.pt', 'start_step_unit': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-All-Bit/resumable-task-v1/run/step_2_unit.pt'}, 'objective': {'mu': 0.02, 'gamma': 0.01, 'payload_weight': 0.8, 'state_weight': 0.2, 'step_size': 0.1, 'cap': 1.0}}
EXPECTED_COST = dict(vae_load=1, decode=6, decoder_vjp=2, encode=0, dit=0, native=0, scheduler=0, codec=0,
    ordinary_chunks=184, gradient_forward_chunks=92, nominal_replay_chunks=92,
    windows=352, chips=5632, metrics=220)
OUTPUT_PARENT = Path('/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-All-Bit-Continue')
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = OUTPUT_PARENT / 'resumable-task-v1-2x010'  # Same logical task on every Run all.
OUTPUT.mkdir(exist_ok=True)
ATTEMPT_OUTPUT = OUTPUT / 'notebook_attempts' / STAMP
ATTEMPT_OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT = OUTPUT / 'run'
WORKSPACE = Path('/content/Video-WM-Local-Joint-All-Bit-Continue-' + STAMP)
PYTHON = sys.executable
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    os.replace(temporary, path)

def original_record_failure(stage, exc):
    row = dict(stage=stage, reason=f'{type(exc).__name__}: {exc}', traceback=''.join(traceback.format_exception(type(exc), exc, exc.__traceback__)), retry=False, output=str(OUTPUT), expected_cost=EXPECTED_COST)
    write_json(ATTEMPT_OUTPUT / 'notebook_failure.json', row)

def logged(command, stage, *, cwd=None, env=None, check=True):
    child = None
    returncode = None
    primary = None
    primary_tb = None
    cleanup_errors = []

    def cleanup_error(label, exc):
        cleanup_errors.append(label + ': ' + repr(exc))
    try:
        with (ATTEMPT_OUTPUT / 'execution.log').open('a', encoding='utf-8') as log:
            line = 'STAGE ' + stage + ' COMMAND ' + repr(command) + '\n'
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
            group = {'start_new_session': True} if os.name == 'posix' else {}
            child = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, **group)
            try:
                for line in child.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                returncode = child.wait()
            except BaseException as exc:
                primary = exc
                primary_tb = exc.__traceback__
            finally:
                if child is not None:
                    if os.name == 'posix':
                        try:
                            os.killpg(child.pid, signal.SIGTERM)
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('group SIGTERM', exc)
                        deadline = time.monotonic() + 5.0
                        while time.monotonic() < deadline:
                            try:
                                os.killpg(child.pid, 0)
                            except ProcessLookupError:
                                break
                            except BaseException as exc:
                                cleanup_error('group probe', exc)
                                break
                            time.sleep(0.05)
                        try:
                            os.killpg(child.pid, signal.SIGKILL)
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('group SIGKILL', exc)
                    elif child.poll() is None:
                        try:
                            child.terminate()
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('terminate', exc)
                    try:
                        returncode = child.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        try:
                            child.kill()
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('kill', exc)
                        try:
                            returncode = child.wait(timeout=10)
                        except BaseException as exc:
                            cleanup_error('wait after kill', exc)
                    except BaseException as exc:
                        cleanup_error('wait', exc)
                    if child.stdout is not None:
                        try:
                            child.stdout.close()
                        except BaseException as exc:
                            cleanup_error('stdout close', exc)
            if primary is None and (not cleanup_errors):
                log.write('EXIT ' + str(returncode) + '\n')
                log.flush()
    except BaseException as exc:
        if primary is None:
            primary = exc
            primary_tb = exc.__traceback__
        else:
            cleanup_error('log wrapper', exc)
    if primary is None and cleanup_errors:
        primary = RuntimeError('process cleanup failed: ' + '; '.join(cleanup_errors))
        primary_tb = primary.__traceback__
    if primary is None and check and returncode:
        primary = subprocess.CalledProcessError(returncode, command)
        primary_tb = primary.__traceback__
    if primary is not None:
        if cleanup_errors and hasattr(primary, 'add_note'):
            primary.add_note('secondary cleanup errors: ' + '; '.join(cleanup_errors))
        try:
            record_failure(stage, primary)
        except BaseException as record_error:
            if hasattr(primary, 'add_note'):
                primary.add_note('failure record error: ' + repr(record_error))
        raise primary.with_traceback(primary_tb)
    return returncode

def record_failure(stage, exc):
    original_record_failure(stage, exc)
    fixed_slots.update(status='ENGINEERING_FAILURE', reason=repr(exc))
    write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
    if (RUN_OUTPUT / 'result.json').exists():
        from runtime.wan.local_joint_all_bit_continue_v1 import finalize_interrupted
        finalize_interrupted(RUN_OUTPUT, 'notebook takeover after child cleanup: ' + repr(exc))

fixed_slots = dict(status='SETUP_STARTED', expected_cost=EXPECTED_COST,
    views={name: dict(status='NOT_RUN', expected_windows=88, expected_chips=1408, expected_metrics=55)
           for name in ('BASE','START_SAVED','CONT_MID','CONT_FINAL')})
write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
if (OUTPUT / 'fixed_config.json').exists():
    FIXED_CONFIG = json.loads((OUTPUT / 'fixed_config.json').read_text())
else:
    write_json(OUTPUT / 'fixed_config.json', FIXED_CONFIG)
write_json(ATTEMPT_OUTPUT / 'setup_receipt.json', dict(config=FIXED_CONFIG, expected_cost=EXPECTED_COST,
    output=str(OUTPUT), python=sys.version, scientific_pass=False, automatic_retry=False))
print('Fixed ALLBIT-CONTINUE-2x010 output:', OUTPUT, flush=True)

In [ ]:
import io, urllib.request, zipfile
SOURCE_URL = 'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/' + SOURCE_REF + '/notebooks/local_joint_all_bit_continue_v1_portable_source.zip'
try:
    with urllib.request.urlopen(SOURCE_URL) as response:
        package_bytes = response.read()
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(package_bytes)) as archive:
        for member in archive.infolist():
            target = (WORKSPACE / member.filename).resolve()
            if not target.is_relative_to(WORKSPACE.resolve()):
                raise RuntimeError('unsafe source archive path')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(member))
    # A repeated Run all may retain packages from an earlier extracted workspace.
    import importlib
    for module_name in list(sys.modules):
        if module_name.split('.', 1)[0] in ('main', 'runtime', 'experiments'):
            sys.modules.pop(module_name, None)
    importlib.invalidate_caches()
    sys.path.insert(0, str(WORKSPACE))
    print('Source:', SOURCE_URL, 'Workspace:', WORKSPACE, flush=True)
    try:
        write_json(ATTEMPT_OUTPUT / 'source_receipt.json', dict(url=SOURCE_URL, workspace=str(WORKSPACE)))
    except Exception as exc:
        print('Source note unavailable:', repr(exc), flush=True)
except BaseException as exc:
    record_failure('SOURCE_DOWNLOAD', exc)
    raise

In [ ]:
from runtime.wan.local_joint_all_bit_continue_v1 import model_work_remaining
try:
    NEEDS_MODEL = model_work_remaining(RUN_OUTPUT)
    probe_path = OUTPUT / 'dependency_probe.json'
    probe_lines = [
        "import json, pathlib, sys, torch, numpy, PIL",
        "row = dict(python=sys.version, torch=str(torch.__version__), model_work_remaining=" + repr(NEEDS_MODEL) + ")",
    ]
    if NEEDS_MODEL:
        probe_lines += ["from diffusers import AutoencoderKLWan"]
        if str(FIXED_CONFIG.get('device', 'cuda')).startswith('cuda'):
            probe_lines += ["assert torch.cuda.is_available(), 'CUDA is needed for unfinished decode/VJP'",
                            "row.update(cuda_available=True, device=torch.cuda.get_device_name(0))"]
    probe_lines += ["pathlib.Path(" + repr(str(probe_path)) + ").write_text(json.dumps(row))"]
    probe_code = '\n'.join(probe_lines)
    probe_returncode = logged([PYTHON, '-u', '-c', probe_code], 'DEPENDENCY_PROBE', check=False)
    if probe_returncode:
        packages = ['torch', 'numpy', 'pillow']
        if NEEDS_MODEL:
            packages += ['diffusers', 'transformers', 'accelerate', 'safetensors', 'huggingface-hub', 'sentencepiece', 'ftfy']
        logged([PYTHON, '-m', 'pip', 'install', *packages], 'DEPENDENCY_REPAIR')
        logged([PYTHON, '-u', '-c', probe_code], 'DEPENDENCY_REPROBE')
    dependency_check_returncode = logged([PYTHON, '-m', 'pip', 'check'], 'DEPENDENCY_REPORT', check=False)
    environment_receipt = json.loads(probe_path.read_text())
    environment_receipt.update(dependency_check_returncode=dependency_check_returncode,
        source_and_versions_are_location_notes=True, checkpoint_interval_seconds=60)
    write_json(ATTEMPT_OUTPUT / 'environment_receipt.json', environment_receipt)
    print('Remaining model work:', NEEDS_MODEL, '; task:', OUTPUT, flush=True)
except BaseException as exc:
    record_failure('ENVIRONMENT_SETUP', exc)
    raise

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token: os.environ['HF_TOKEN'] = hf_token
except Exception as exc:
    print('Optional HF token unavailable:', type(exc).__name__, flush=True)

run_returncode = None
try:
    command = [PYTHON, '-u', '-m', 'experiments.wan_state_clock.local_joint_all_bit_continue_v1_run',
               '--config', str(OUTPUT / 'fixed_config.json'), '--output', str(RUN_OUTPUT)]
    run_returncode = logged(command, 'FIXED_ALL_BIT_CONTINUE', cwd=WORKSPACE, check=False)
    from runtime.wan.local_joint_all_bit_continue_v1 import finalize_interrupted
    # Takeover is report-only and follows child/process-group reap in logged().
    finalize_interrupted(RUN_OUTPUT, 'child exited with return code ' + str(run_returncode))
    result_path = RUN_OUTPUT / 'result.json'
    result = json.loads(result_path.read_text()) if result_path.exists() else None
    write_json(ATTEMPT_OUTPUT / 'execution_receipt.json', dict(returncode=run_returncode,
        status=None if result is None else result['status'],
        actual_counts=None if result is None else result['counts'], expected_cost=EXPECTED_COST,
        scientific_pass=False, automatic_retry=False))
    if run_returncode != 0 or result is None or result['status'] != 'COMPLETE':
        raise RuntimeError('ALLBIT-CONTINUE-2x010 engineering interruption; retain entire output: ' + str(OUTPUT))
    fixed_slots.update(status='SUPERSEDED_BY_RUN_RESULT', result='run/result.json')
    write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
except BaseException as exc:
    try: record_failure('FIXED_ALL_BIT_CONTINUE', exc)
    except BaseException as record_error:
        if hasattr(exc, 'add_note'): exc.add_note('failure record error: ' + repr(record_error))
    raise

In [ ]:
result = json.loads((RUN_OUTPUT / 'result.json').read_text())
print('Engineering status:', result['status'])
print('Cumulative physical calls:', result['counts'])
print('This Run all:', result.get('current_attempt_counts', {}))
print('Ordinary decode chunks:', result['ordinary_chunks'])
for name, replay in result['replay'].items():
    print('Gradient ledger:', name, replay['counts'], replay.get('boundary_storage', {}))
for name, view in result['views'].items():
    print(name, view['status'], 'G=', view.get('state_gap'),
          'positive bits=', sum(x is not None and x>0 for x in view.get('margins', [])), '/32',
          'reason=', view.get('reason'))
print(result['ceiling'])
print('Return the entire output directory, including failures:', OUTPUT)